In [10]:
import pandas as pd
import numpy as np

In [11]:
df_train = pd.read_csv("../data/raw/Train.csv")
df_demo = pd.read_parquet("../data/raw/demographics_clean.parquet")
df_raw = df_train.merge(df_demo, on="UniqueID", how="left")

In [12]:
df_raw["Age"] = (pd.Timestamp.today() - pd.to_datetime(df_raw["BirthDate"])).dt.days // 365
df_raw.drop(columns=["BirthDate"], inplace=True)

In [13]:
df_raw = df_raw[df_raw["Age"].between(18, 100)].reset_index(drop=True)

In [14]:
df_raw["AnnualGrossIncome"] = df_raw["AnnualGrossIncome"].fillna(
    df_raw["AnnualGrossIncome"].median()
)

for col in ["ResidentialCityName", "CountryCodeNationality",
            "LowIncomeFlag", "CustomerOnboardingChannel",
            "CustomerBankingType", "Gender"]:
    df_raw[col] = df_raw[col].fillna(df_raw[col].mode()[0])

In [15]:
df_raw["AnnualGrossIncome_log"] = np.log1p(df_raw["AnnualGrossIncome"])
df_raw.drop(columns=["AnnualGrossIncome"], inplace=True)

In [16]:
df_processed = pd.get_dummies(df_raw, columns=[
    "Gender", "IncomeCategory", "CustomerStatus", "ClientType",
    "MaritalStatus", "OccupationCategory", "IndustryCategory",
    "CustomerBankingType", "CustomerOnboardingChannel",
    "LowIncomeFlag", "ContactPreference",
    "ResidentialCityName", "CountryCodeNationality",
    "CertificationTypeDescription"
], drop_first=True)

In [17]:
df_processed.drop(columns=["UniqueID"], inplace=True)

In [ ]:
assert df_processed.isna().sum().sum() == 0, "NaN restants"
print(f"Shape final : {df_processed.shape}")
print(df_processed.dtypes.value_counts())

Shape final : (8250, 599)
bool       596
int64        2
float64      1
Name: count, dtype: int64


In [19]:
df_processed.to_parquet("../data/processed/data_processed.parquet", index=False)